<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/main/STEP_7_SARSA_IMPLEMENTATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **STEP 7: SARSA IMPLEMENTATION**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import joblib
import time

# 1. Precompute arrays
# -----------------------------

states = train_df["state_id"].astype(int).to_numpy()
actions_observed = train_df["action_id"].astype(int).to_numpy()
rewards = train_df["reward"].astype(float).to_numpy()

current_states = states[:-1]
next_states = states[1:]

n_steps = len(current_states)

print("Training transitions:", n_steps)

# -----------------------------
# 2. Precompute valid actions by state
# -----------------------------

valid_actions_array = {}

# Use n_states instead of n_states_sarsa
for s in range(n_states):
    acts = train_df.loc[
        train_df["state_id"] == s,
        "action_id"
    ].unique()

    valid_actions_array[s] = np.array(acts, dtype=int)

# -----------------------------
# 3. Faster helper functions
# -----------------------------

def fast_choose_action(state, epsilon):
    valid_actions = valid_actions_array.get(state)

    if valid_actions is None or len(valid_actions) == 0:
        return None

    if np.random.random() < epsilon:
        return np.random.choice(valid_actions)

    # Use Q instead of Q_sarsa
    q_vals = Q[state, valid_actions]
    return valid_actions[np.argmax(q_vals)]


def fast_expected_q(next_state, epsilon):
    valid_actions = valid_actions_array.get(next_state)

    if valid_actions is None or len(valid_actions) == 0:
        return 0.0

    # Use Q instead of Q_sarsa
    q_vals = Q[next_state, valid_actions]

    best_idx = np.argmax(q_vals)

    n_valid = len(valid_actions)

    probs = np.full(n_valid, epsilon / n_valid)
    probs[best_idx] += 1.0 - epsilon

    return np.dot(probs, q_vals)

# -----------------------------
# 4. Faster training parameters
# -----------------------------

alpha = 0.15
alpha_decay = 0.9995
alpha_min = 0.03

gamma = 0.75

epsilon = 1.0
epsilon_decay = 0.997
epsilon_min = 0.03

episodes = 300   # start lower for full dataset

# -----------------------------
# 5. Fast Expected SARSA loop
# -----------------------------

start_time = time.time()

# Use n_states instead of n_states_sarsa
previous_policy = np.full(n_states, -1)

for episode in range(episodes):
    total_td_error = 0.0
    updates = 0

    for i in range(n_steps):
        state = current_states[i]
        next_state = next_states[i]

        action = fast_choose_action(state, epsilon)

        if action is None:
            continue

        reward = rewards[i]
        expected_next_q = fast_expected_q(next_state, epsilon)

        # Use Q instead of Q_sarsa
        old_q = Q[state, action]

        target = reward + gamma * expected_next_q
        td_error = target - old_q

        # Use Q instead of Q_sarsa
        Q[state, action] += alpha * td_error

        total_td_error += abs(td_error)
        updates += 1

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    alpha = max(alpha_min, alpha * alpha_decay)

    # Current greedy policy
    # Use n_states instead of n_states_sarsa
    current_policy = np.full(n_states, -1)

    for s in range(n_states):
        valid_actions = valid_actions_array.get(s)

        if valid_actions is not None and len(valid_actions) > 0:
            # Use Q instead of Q_sarsa
            current_policy[s] = valid_actions[
                np.argmax(Q[s, valid_actions])
            ]

    if episode == 0:
        policy_change_pct = 100.0
    else:
        policy_change_pct = (
            np.mean(current_policy != previous_policy) * 100
        )

    previous_policy = current_policy.copy()

    avg_td_error = total_td_error / max(updates, 1)

    if episode % 25 == 0:
        elapsed = time.time() - start_time
        print(
            f"Episode {episode}: "
            f"Avg TD Error = {avg_td_error:.6f}, "
            f"Policy Change = {policy_change_pct:.2f}%, "
            f"Epsilon = {epsilon:.4f}, "
            f"Alpha = {alpha:.4f}, "
            f"Elapsed = {elapsed/60:.2f} min"
        )

print("Fast Expected SARSA training complete.")
print(f"Total runtime: {(time.time() - start_time)/60:.2f} minutes")

# Extract Policy

In [ ]:
sarsa_policy_action_ids = []

for s in range(n_states):
    valid_actions = valid_actions_array.get(s, [])

    if len(valid_actions) == 0:
        # Fallback if no valid actions for the state are found, pick the best from all possible actions
        # This assumes Q is correctly shaped for all states and actions
        best_action = np.argmax(Q[s])
    else:
        best_action = valid_actions[np.argmax(Q[s, valid_actions])]

    sarsa_policy_action_ids.append(best_action)

sarsa_policy_df = pd.DataFrame({
    "state_id": np.arange(n_states),
    "action_id": sarsa_policy_action_ids,
    "recommended_action": action_encoder.inverse_transform(sarsa_policy_action_ids)
})

split_cols = sarsa_policy_df["recommended_action"].str.split(r" \| ", expand=True)

if split_cols.shape[1] == 3:
    sarsa_policy_df["Recommended_Fulfillment_Region"] = split_cols[0]
    sarsa_policy_df["Recommended_Route"] = split_cols[1]
    sarsa_policy_df["Recommended_Shipping_Mode"] = split_cols[2]
elif split_cols.shape[1] == 2:
    sarsa_policy_df["Recommended_Route"] = split_cols[0]
    sarsa_policy_df["Recommended_Shipping_Mode"] = split_cols[1]
else:
    sarsa_policy_df["Recommended_Shipping_Mode"] = sarsa_policy_df["recommended_action"]

display(sarsa_policy_df.head())

#Evaluate Outcomes

In [ ]:
# Profit-protected SARSA evaluation

eval_df = test_df.copy()

# Historical action outcomes from training data
action_outcomes_sarsa = train_df.groupby("action")[[
    "Order Profit Per Order",
    "Days for shipping (real)",
    "Late_delivery_risk"
]].mean()

# Merge SARSA recommendation onto test data
sarsa_eval_df = eval_df.merge(
    sarsa_policy_df[["state_id", "recommended_action"]],
    on="state_id",
    how="left"
)

# Add recommended action outcomes
sarsa_eval_df = sarsa_eval_df.merge(
    action_outcomes_sarsa,
    left_on="recommended_action",
    right_index=True,
    how="left",
    suffixes=["_historical", "_policy"]
)

# Fill missing policy outcomes with historical row values
for col in ["Order Profit Per Order", "Days for shipping (real)", "Late_delivery_risk"]:
    sarsa_eval_df[col + "_policy"] = sarsa_eval_df[col + "_policy"].fillna(
        sarsa_eval_df[col + "_historical"]
    )

# Profit-protection rule:
# only accept SARSA recommendation if profit is not worse
accept_policy = (
    sarsa_eval_df["Order Profit Per Order_policy"] >= sarsa_eval_df["Order Profit Per Order_historical"]
)

# Apply fallback to historical action if recommendation loses profit
for col in ["Order Profit Per Order", "Days for shipping (real)", "Late_delivery_risk"]:
    sarsa_eval_df[col + "_final"] = np.where(
        accept_policy,
        sarsa_eval_df[col + "_policy"],
        sarsa_eval_df[col + "_historical"]
    )

historical_profit = sarsa_eval_df["Order Profit Per Order_historical"].mean()
policy_profit = sarsa_eval_df["Order Profit Per Order_final"].mean()

historical_days = sarsa_eval_df["Days for shipping (real)_historical"].mean()
policy_days = sarsa_eval_df["Days for shipping (real)_final"].mean()

historical_late_risk = sarsa_eval_df["Late_delivery_risk_historical"].mean()
policy_late_risk = sarsa_eval_df["Late_delivery_risk_final"].mean()

profit_change_pct = ((policy_profit - historical_profit) / historical_profit) * 100
days_change_pct = ((historical_days - policy_days) / historical_days) * 100
late_risk_change_pct = ((historical_late_risk - policy_late_risk) / historical_late_risk) * 100

print(f"Historical Profit: {historical_profit:.4f}")
print(f"SARSA Policy Profit: {policy_profit:.4f}")
print(f"Profit Change %: {profit_change_pct:.2f}%")

print(f"Historical Days: {historical_days:.4f}")
print(f"SARSA Policy Days: {policy_days:.4f}")
print(f"Delivery Time Reduction %: {days_change_pct:.2f}%")

print(f"Historical Late Risk: {historical_late_risk:.4f}")
print(f"SARSA Policy Late Risk: {policy_late_risk:.4f}")
print(f"Late Risk Reduction %: {late_risk_change_pct:.2f}%")

print("Accepted SARSA recommendations:", accept_policy.sum())
print("Rejected profit-losing recommendations:", (~accept_policy).sum())

if profit_change_pct > 0 and days_change_pct > 0 and late_risk_change_pct > 0:
    print("SARSA goal achieved")
else:
    print("SARSA goal not achieved")